# Step 1: Preprocess Labeled Data

Extract PhoBERT text embeddings and ResNet50 image features from labeled ViFactCheck data.

- Input: `../data/json/labeled_nei_as_real/news_data_vifactcheck_*_labeled.json`
- Output: `../processed_data/hdf5/vifactcheck_{train,dev,test}.h5`

Each HDF5 file contains:
- `text_features`: `[N, 512, 768]` (PhoBERT token embeddings)
- `image_features`: `[N, 2048]` (ResNet50 features)
- `labels`: `[N]` (0=Real, 1=Fake)

**Prerequisites**: Run `0_verify_labels.ipynb` first.

In [1]:
import torch
print(torch.__version__)  # Should show 2.11.0+cu128
print(torch.cuda.get_arch_list())  # Should include sm_120
x = torch.randn(2, 3).cuda()
print(x @ x.T)  # Should work without error

2.11.0+cu128
['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
tensor([[ 5.1526, -1.8261],
        [-1.8261,  2.0037]], device='cuda:0')


In [2]:
!nvidia-smi

Thu Apr 16 21:49:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.97                 Driver Version: 595.97         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
| 30%   53C    P0             21W /  180W |    2476MiB /  16311MiB |      6%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import sys, os, gc
import orjson
import numpy as np
import torch
import h5py
from pathlib import Path
from tqdm import tqdm
from collections import Counter

project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

Project root: g:\My Drive\Thesis_Final\fake-new-detection


In [4]:
# Configuration
CONFIG = {
    "text_model": "vinai/phobert-base",
    "image_model": "resnet50",
    "language": "vi",
    "max_length": 256,  # PhoBERT max_position_embeddings=258, so max usable length is 256
    "batch_size": 32,
    "input_dir": str(project_root / "notebooks" / "data" / "json" / "labeled_nei_as_real"),
    "output_dir": str(project_root / "notebooks" / "processed_data" / "hdf5"),
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Device: {device}")
print(f"Input:  {CONFIG['input_dir']}")
print(f"Output: {CONFIG['output_dir']}")

Device: cuda
Input:  g:\My Drive\Thesis_Final\fake-new-detection\notebooks\data\json\labeled_nei_as_real
Output: g:\My Drive\Thesis_Final\fake-new-detection\notebooks\processed_data\hdf5


In [5]:
# Initialize preprocessor
from src.preprocessing import CombinedPreprocessor

preprocessor = CombinedPreprocessor(
    text_model_name=CONFIG["text_model"],
    image_model_name=CONFIG["image_model"],
    language=CONFIG["language"],
    max_text_length=CONFIG["max_length"],
    device=device,
)
print("Preprocessor ready")

c:\Users\tungh\.conda\envs\fake_news\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 26197.12it/s]
RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect ident

Preprocessor ready


In [ ]:
# Load and prepare each split
from PIL import Image

input_dir = Path(CONFIG["input_dir"])
image_base_dir = project_root / "notebooks" / "data"  # images are at notebooks/data/jpg/...

# Create zero placeholder for missing images
placeholder_dir = Path(CONFIG["output_dir"]) / "placeholder_images"
placeholder_dir.mkdir(exist_ok=True)
zero_path = placeholder_dir / "zero_placeholder.jpg"
if not zero_path.exists():
    Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8)).save(zero_path)


def load_and_prepare(split_name):
    """Load labeled JSON, extract texts/images/labels."""
    json_file = input_dir / f"news_data_vifactcheck_{split_name}_labeled.json"
    with open(json_file, "rb") as f:
        articles = orjson.loads(f.read())

    texts, image_paths, labels = [], [], []
    for article in articles:
        text = (
            article.get("text", "")
            or article.get("content", "")
            or article.get("title", "")
        )
        if not text.strip():
            continue

        texts.append(text.strip())
        labels.append(int(article["label"]))

        images = article.get("images", [])
        if images and len(images) > 0:
            if isinstance(images[0], dict):
                img_rel = (
                    images[0].get("folder_path", "")
                    or images[0].get("path", "")
                )
            else:
                img_rel = str(images[0])

            # Resolve relative path against image base directory
            if img_rel:
                img_abs = image_base_dir / img_rel
                img_path = str(img_abs) if img_abs.exists() else str(zero_path)
            else:
                img_path = str(zero_path)
            image_paths.append(img_path)
        else:
            image_paths.append(str(zero_path))

    dist = Counter(labels)
    n_real_img = sum(1 for p in image_paths if str(zero_path) not in p)
    n_zero = sum(1 for p in image_paths if str(zero_path) in p)
    print(f"{split_name}: {len(texts)} samples, labels={dict(dist)}, real_images={n_real_img}, missing_images={n_zero}")
    assert len(set(labels)) >= 2, f"ERROR: {split_name} has only {len(set(labels))} class!"
    return texts, image_paths, labels


splits = {}
for name in ["train", "dev", "test"]:
    splits[name] = load_and_prepare(name)

print("\nAll splits loaded.")

In [7]:
# Batch preprocessing for each split
BATCH_SIZE = CONFIG["batch_size"]

processed = {}

for split_name, (texts, image_paths, labels) in splits.items():
    total = len(texts)
    n_batches = (total + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"\nProcessing {split_name}: {total} samples, {n_batches} batches")

    all_text, all_image = [], []

    for i in tqdm(range(n_batches), desc=split_name):
        s, e = i * BATCH_SIZE, min((i + 1) * BATCH_SIZE, total)

        text_feat = preprocessor.text_preprocessor.extract_token_embeddings(texts[s:e])
        img_feat = preprocessor.image_preprocessor.extract_features(image_paths[s:e])

        all_text.append(text_feat)
        all_image.append(img_feat)

        del text_feat, img_feat
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    processed[split_name] = {
        "text": np.vstack(all_text),
        "image": np.vstack(all_image),
        "labels": np.array(labels),
    }
    print(f"  text={processed[split_name]['text'].shape}, image={processed[split_name]['image'].shape}")

    del all_text, all_image
    gc.collect()

print("\nAll splits preprocessed.")


Processing train: 3376 samples, 106 batches


train: 100%|██████████| 106/106 [01:59<00:00,  1.13s/it]


  text=(3376, 256, 768), image=(3376, 2048)

Processing dev: 465 samples, 15 batches


dev: 100%|██████████| 15/15 [00:19<00:00,  1.32s/it]


  text=(465, 256, 768), image=(465, 2048)

Processing test: 973 samples, 31 batches


test: 100%|██████████| 31/31 [00:43<00:00,  1.40s/it]


  text=(973, 256, 768), image=(973, 2048)

All splits preprocessed.


In [8]:
# Save each split as separate HDF5
output_dir = Path(CONFIG["output_dir"])

for split_name, data in processed.items():
    text_feat = data["text"]
    image_feat = data["image"]
    labels = data["labels"]
    n = len(labels)

    hdf5_path = output_dir / f"vifactcheck_{split_name}.h5"
    chunk_size = min(64, n)

    with h5py.File(hdf5_path, "w") as f:
        f.create_dataset(
            "text_features", data=text_feat,
            chunks=(chunk_size, text_feat.shape[1], text_feat.shape[2]),
            compression="gzip", compression_opts=4,
        )
        f.create_dataset(
            "image_features", data=image_feat,
            chunks=(chunk_size, image_feat.shape[1]),
            compression="gzip", compression_opts=4,
        )
        f.create_dataset("labels", data=labels)
        f.attrs["n_samples"] = n
        f.attrs["text_shape"] = text_feat.shape[1:]
        f.attrs["image_shape"] = image_feat.shape[1:]
        f.attrs["label_distribution"] = str(dict(Counter(labels.tolist())))

    size_mb = hdf5_path.stat().st_size / (1024 * 1024)
    print(f"{split_name}: {n} samples, labels={dict(Counter(labels.tolist()))}, size={size_mb:.1f} MB")

# Cleanup
del processed
gc.collect()

print(f"\nHDF5 files saved to: {output_dir}")
print("Proceed to Step 2a or 2b for training.")

train: 3376 samples, labels={1: 3113, 0: 263}, size=2004.5 MB
dev: 465 samples, labels={1: 243, 0: 222}, size=281.4 MB
test: 973 samples, labels={1: 612, 0: 361}, size=568.9 MB

HDF5 files saved to: g:\My Drive\Thesis_Final\fake-new-detection\notebooks\processed_data\hdf5
Proceed to Step 2a or 2b for training.


In [9]:
# Final verification: read back and check
output_dir = Path(CONFIG["output_dir"])

for name in ["train", "dev", "test"]:
    path = output_dir / f"vifactcheck_{name}.h5"
    with h5py.File(path, "r") as f:
        labels = f["labels"][:]
        text_shape = f["text_features"].shape
        image_shape = f["image_features"].shape
    dist = Counter(labels.tolist())
    print(f"{name}: text={text_shape}, image={image_shape}, labels={dict(dist)}")
    assert len(dist) >= 2, f"ERROR: {name} has only {len(dist)} class!"

print("\nAll HDF5 files verified. Ready for training.")

train: text=(3376, 256, 768), image=(3376, 2048), labels={1: 3113, 0: 263}
dev: text=(465, 256, 768), image=(465, 2048), labels={1: 243, 0: 222}
test: text=(973, 256, 768), image=(973, 2048), labels={1: 612, 0: 361}

All HDF5 files verified. Ready for training.
